<div style="text-align:center; padding:18px 0 6px 0"></div>

<div style="text-align:center; color:#8b949e; font-size:12px; padding-bottom:8px">Copyright © QPerfect &middot; Released under the MIT License &middot; <a href="https://qperfect.io">qperfect.io</a></div>

---

# Bond-dimension fidelity sweep — scaling QAOA on MIMIQ MPS

We've solved a 4-city TSP end-to-end. Bigger problems are where MPS
shines: it scales polynomially in qubit count and stays exact as long as
the entanglement fits within a fixed bond dimension χ. **Higher χ ⇒
better fidelity, more memory, more time.** Reordering qubits to
respect the problem's interaction structure can also reduce the χ
needed for a given fidelity.

This notebook fixes the QAOA angles (no optimisation — straight linear
ramp at depth `P_LAYERS`, default 8) and submits the ansatz to MIMIQ at
several bond dimensions, with and without `reorderqubits`. The cloud
returns a **truncation-fidelity** estimate per run; we plot it against
χ on log scale to see at what χ each problem stabilises.

`P_LAYERS = 8` is a sweet spot for a *demonstrative* curve on these
dense QUBOs — entanglement is non-trivial at low χ and converges by
χ = 256. p = 2 is too easy (everything saturates at high fidelity);
p ≥ 16 is too hard (everything bottoms out below χ = 256 because
Lucas-2014 / full-one-hot couplings are dense, unlike the 3-regular
MaxCut where p = 20 is feasible).

Two problems, both built from the canonical Strasbourg places:

- **TSP-7** — 7 markets, Lucas-2014 one-hot, **49 qubits**
- **VRP-7/3** — 7 customers + Gare-Centrale depot, K=3 vehicles, full
  one-hot encoding, **63 qubits**

> First run hits the cloud ~20 times (5 χ values × 2 reorder options ×
> 2 problems). Each MIMIQ submit takes a few minutes; total wall-clock
> can be 30–60 min depending on queue. Subsequent runs are offline via
> `mimiq_cache`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import mimiqcircuits as mc
from strasbourg_markets_demo import strasbourg as s, theme
from strasbourg_markets_demo.cache import mimiq_cache
from strasbourg_markets_demo.qaoa import linear_ramp, build_qaoa
from strasbourg_markets_demo.tsp import TSPInstance, to_qubo as tsp_to_qubo
from strasbourg_markets_demo.vrp import VRPInstance, to_qubo as vrp_to_qubo

theme.apply()
RNG_SEED = 2024

# QAOA depth for the sweep. Sweet spot for a demonstrative fidelity-vs-χ
# curve is somewhere around p=4 to p=10:
#   - p=2 produces little entanglement; even χ=16 gives high fidelity (flat plot).
#   - p=8 puts the χ ∈ {16..256} range right across the transition.
#   - p≥16 likely bottoms out (entanglement exceeds χ=256 on these dense QUBOs).
# Lucas-2014 / full-one-hot are dense couplings — much harder than 3-regular
# MaxCut, so don't expect Feeney-style p=20 results here.
P_LAYERS = 8
BOND_DIMS = [16, 32, 64, 128, 256]


## 1. Build the TSP-7 ansatz

Seven markets selected from the canonical Strasbourg list — the five
TSP-demo cities plus Place Benjamin Zix and Place du Temple Neuf.


In [ ]:
TSP7_KEYS = ["kleber", "broglie", "cathedrale_market", "chateau",
             "marche_gayot", "benjamin_zix", "temple_neuf"]
markets7 = [s.find(k) for k in TSP7_KEYS]
tsp_inst = TSPInstance.from_coords(
    coords=s.coords_array(markets7),
    names=[m.name for m in markets7],
    metric="haversine",
)
print(f"TSP-{tsp_inst.n} cities: {[m.name.split()[0] for m in markets7]}")

tsp_qubo = tsp_to_qubo(tsp_inst)
h_t, J_t, c_t = tsp_qubo.to_ising()
scale_t = float(max(np.abs(h_t).max(), np.abs(J_t).max()))
h_t_n, J_t_n = h_t / scale_t, J_t / scale_t
print(f"TSP-7 QUBO -> {tsp_qubo.n_qubits} qubits, {len(tsp_qubo.labels)} labels")


## 2. Build the VRP-7/3 ansatz

Seven customers, three couriers, depot at the Gare. Position dimension
defaults to ⌈7/3⌉ = 3, giving 7 × 3 × 3 = 63 qubits.


In [ ]:
depot = s.find("gare_centrale")
vrp_inst = VRPInstance.from_coords(
    depot_coord=(depot.lat, depot.lon),
    customer_coords=[(m.lat, m.lon) for m in markets7],
    n_vehicles=3,
    names=[depot.name] + [m.name for m in markets7],
    metric="haversine",
)
print(f"VRP-{vrp_inst.n_customers}/{vrp_inst.n_vehicles}: "
      f"capacity (auto)={vrp_inst.capacity}")

vrp_qubo = vrp_to_qubo(vrp_inst)
h_v, J_v, c_v = vrp_qubo.to_ising()
scale_v = float(max(np.abs(h_v).max(), np.abs(J_v).max()))
h_v_n, J_v_n = h_v / scale_v, J_v / scale_v
print(f"VRP-7/3 QUBO -> {vrp_qubo.n_qubits} qubits")


## 3. Linear-ramp QAOA ansatz (no optimisation)

Standard QAOA ansatz with the adiabatic-style linear ramp at depth
`P_LAYERS`. We're not optimising — angles are fixed — because the
goal here is to characterise *the simulator*, not the algorithm.
More layers ⇒ more entanglement ⇒ larger χ needed for fidelity.

`linear_ramp` and `build_qaoa` come from `strasbourg_markets_demo.qaoa`
(same definitions as in `qaoa_demo.ipynb` §3 — see that notebook for
the inline implementation).


In [ ]:
gammas, betas = linear_ramp(P_LAYERS)
tsp_circ = build_qaoa(h_t_n, J_t_n, gammas, betas)
vrp_circ = build_qaoa(h_v_n, J_v_n, gammas, betas)
print(f"TSP-7  circuit: {len(tsp_circ)} instructions, depth {tsp_circ.depth()}")
print(f"VRP-7/3 circuit: {len(vrp_circ)} instructions, depth {vrp_circ.depth()}")


## 4. Connect to MIMIQ

In [ ]:
conn = mc.MimiqConnection(mc.QPERFECT_CLOUD)
conn.connect()
print(f"connected to {mc.QPERFECT_CLOUD}")


## 5. Sweep χ × reorder, both problems

Each cell submission is wrapped in `mimiq_cache` so repeat runs are
offline. The cache key is `"fidsweep-<problem>-r<reorder>-bd<χ>"`,
unique per cell. We record both the **mean truncation fidelity**
(`res.fidelities` averaged over shots) and the **server wall-clock
total** (`res.timings["total"]`, in seconds) — the timing comes from
the original cloud run, so cached replays still report the real
computation time.


In [ ]:
problems = {
    "tsp7":  (tsp_circ, "TSP-7  (49 qubits)"),
    "vrp73": (vrp_circ, "VRP-7/3 (63 qubits)"),
}
reorder_options = [None, True]   # None = no reorder; True = MIMIQ-default reordering

# fidelities[(problem_key, reorder, bd)] -> mean fidelity
# wall_times[(problem_key, reorder, bd)] -> server-side total (seconds)
fidelities: dict[tuple, float] = {}
wall_times: dict[tuple, float] = {}

for prob_key, (circ, title) in problems.items():
    for reorder in reorder_options:
        for bd in BOND_DIMS:
            cache_key = f"fidsweep-{prob_key}-r{reorder}-bd{bd}"
            with mimiq_cache(conn, key=cache_key, verbose=False):
                job = conn.submit(
                    circ,
                    algorithm="mps",
                    bonddim=bd,
                    nsamples=64,
                    seed=RNG_SEED,
                    reorderqubits=reorder,
                    label=cache_key,
                )
                res = conn.get_result(job)
            fid = float(np.mean(res.fidelities)) if res.fidelities else float("nan")
            t_total = float(res.timings.get("total", float("nan"))) if res.timings else float("nan")
            fidelities[(prob_key, reorder, bd)] = fid
            wall_times[(prob_key, reorder, bd)] = t_total
            print(f"{title}  reorder={str(reorder):<5s}  bd={bd:>3d}  "
                  f"fidelity = {fid:.4f}    total = {t_total:7.2f} s")


## 6. Plot fidelity and wall-clock vs bond dimension

Top row: MPS truncation fidelity (we want this near 1 — flat-topped is
ideal). Bottom row: server-side wall-clock for the same submissions
(grows with χ and again with the gate-count, log-y to read across the
range).


In [ ]:
from strasbourg_markets_demo.strasbourg import place_legend_outside

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

for col, (prob_key, (_, title)) in enumerate(problems.items()):
    ax_fid = axes[0, col]
    ax_t   = axes[1, col]
    for reorder, color, marker, lbl in [
        (None, theme.PALETTE["cyan"], "o", "reorderqubits=None"),
        (True, theme.PALETTE["red"],  "s", "reorderqubits=True"),
    ]:
        fids = [fidelities[(prob_key, reorder, bd)] for bd in BOND_DIMS]
        times = [wall_times[(prob_key, reorder, bd)] for bd in BOND_DIMS]
        ax_fid.plot(BOND_DIMS, fids,  marker + "-", color=color, lw=2, ms=7, label=lbl)
        ax_t.plot(  BOND_DIMS, times, marker + "-", color=color, lw=2, ms=7, label=lbl)

    ax_fid.set_title(title)
    ax_fid.set_ylabel("MPS truncation fidelity")
    ax_fid.set_ylim(0, 1.02)
    place_legend_outside(ax_fid)

    ax_t.set_xscale("log", base=2)
    ax_t.set_yscale("log")
    ax_t.set_xticks(BOND_DIMS)
    ax_t.set_xticklabels(BOND_DIMS)
    ax_t.set_xlabel("bond dimension χ")
    ax_t.set_ylabel("server total wall-clock (s)")
    place_legend_outside(ax_t)

plt.tight_layout()
plt.show()


In [ ]:
# Tabular summary — fidelity then wall-clock.
def _fmt_row(d, prob_key, reorder, fmt):
    return "  ".join(format(d[(prob_key, reorder, bd)], fmt) for bd in BOND_DIMS)

print("=== fidelity ===")
print(f"{'problem':<22s}  {'reorder':<8s}  " + "  ".join(f"χ={bd:>4d}" for bd in BOND_DIMS))
print("-" * (22 + 8 + len(BOND_DIMS) * 8))
for prob_key, (_, title) in problems.items():
    for reorder in reorder_options:
        print(f"{title:<22s}  {str(reorder):<8s}  {_fmt_row(fidelities, prob_key, reorder, '7.4f')}")

print()
print("=== server wall-clock total (s) ===")
print(f"{'problem':<22s}  {'reorder':<8s}  " + "  ".join(f"χ={bd:>4d}" for bd in BOND_DIMS))
print("-" * (22 + 8 + len(BOND_DIMS) * 8))
for prob_key, (_, title) in problems.items():
    for reorder in reorder_options:
        print(f"{title:<22s}  {str(reorder):<8s}  {_fmt_row(wall_times, prob_key, reorder, '7.2f')}")


## Summary

- TSP-7 (49 qubits) and VRP-7/3 (63 qubits) are well into MPS-only
  territory — statevector simulation would need $2^{49} \cdot 16 \approx
  9$ PB and $2^{63} \cdot 16 \approx 150$ EB respectively.
- The fidelity-vs-χ curves show **how much truncation actually costs**.
  For QAOA at p=2 with the linear ramp the entanglement is mild and
  fidelity climbs quickly; expect the high-χ end of the curve to be
  near 1.0 on both problems.
- Compare reorder=None against reorder=True: a good qubit ordering
  should let you reach the same fidelity at a smaller χ. The ratio
  varies with problem structure — the routing graphs here are sparse
  enough that you should see a measurable gap at the lowest χ values.
- This is a great slide-able plot for the talk: same y-axis, two
  problems, two curves each. The MPS-MIMIQ pitch lands when the
  audience sees fidelity ≈ 1 at χ = 256 on a 63-qubit circuit that
  classical statevector simulation can't touch.


## Appendix — flush this notebook's cache

The fidelity sweep stores **a lot** of cache files — one per
(problem, reorder, χ) cell — all under the `fidsweep-` prefix. The
cell below — guarded by `if False:` — wipes them and only them.
Caches from `qaoa_demo.ipynb` and `qaoa_server_side.ipynb` are
untouched.


In [ ]:
if False:
    from strasbourg_markets_demo.cache import clear_cache
    n = clear_cache(prefix="fidsweep-")
    print(f"removed {n} cached entr{'y' if n == 1 else 'ies'} for this notebook")



---

<div style="text-align:center; padding:18px 0 6px 0"></div>

<div style="text-align:center; color:#8b949e; font-size:12px; padding-bottom:8px">Copyright © QPerfect &middot; Released under the MIT License &middot; <a href="https://qperfect.io">qperfect.io</a></div>